# Compute Homological Scaffolds — MSC Dataset

Read MSC resting-state `.mat` files directly, compute Pearson correlation → distance → persistent homology (Ripserer.jl), and save **frequency** and **persistence** scaffolds as `.npz`.

Parameters:
- Atlas: `schaefer400-yeo17`
- Pipeline: `regress_36pNS`
- Task: `rest`
- 10 subjects × 10 sessions = 100 scaffold pairs

In [ ]:
using MAT
using Statistics
using Ripserer
using ProgressMeter
using NPZ

## Parameters

In [ ]:
data_dir   = get(ENV, "TOPOFINGER_MSC_ROOT", "./data/MSC") * "/"
atlas      = "schaefer400-yeo17"
pipeline   = "regress_36pNS"
task       = "rest"

output_dir = joinpath(data_dir, "scaffolds", "whole")
mkpath(joinpath(output_dir, "freq"))
mkpath(joinpath(output_dir, "pers"))

subjects = sort(filter(d -> startswith(d, "sub-MSC"), readdir(data_dir)))
println("Subjects: ", subjects)
println("Output dir: ", output_dir)

## Build list of (subject, session, filepath) tuples

In [ ]:
jobs = Tuple{String, Int, String}[]

for sub in subjects
    sub_dir = joinpath(data_dir, sub)
    for ses_idx in 0:9
        ses_str = lpad(ses_idx + 1, 2, '0')  # 01..10
        fname = "$(sub).ses-func$(ses_str)_task-$(task)_out.$(atlas).$(pipeline).mat"
        fpath = joinpath(sub_dir, fname)
        if isfile(fpath)
            push!(jobs, (sub, ses_idx, fpath))
        else
            @warn "Missing: $fname"
        end
    end
end

println("Total jobs: ", length(jobs))

## Compute scaffolds

For each (subject, session):
1. Load time series from `.mat` → `parcel_time[1,1]` (Julia 1-indexed)
2. Compute Pearson correlation matrix
3. Convert to distance: `d = 1 - r`
4. Run Ripserer with H1 representatives
5. Build frequency scaffold (count of generators per edge) and persistence scaffold (sum of persistence per edge)
6. Save as `.npz`

In [ ]:
p = ProgressMeter.Progress(length(jobs))

Threads.@threads for (sub, ses_idx, fpath) in jobs
    out_freq = joinpath(output_dir, "freq", "$(sub)_s$(ses_idx)")
    out_pers = joinpath(output_dir, "pers", "$(sub)_s$(ses_idx)")

    # Skip if already computed
    if isfile(out_freq) && isfile(out_pers)
        ProgressMeter.next!(p)
        continue
    end

    # 1. Load time series
    mat = matread(fpath)
    ts = mat["parcel_time"][1, 1]  # (timepoints × regions)

    # 2. Correlation matrix
    pms = cor(ts)
    num_regions = size(pms, 2)

    # 3. Distance matrix
    dist_corrs = 1.0 .- pms
    dist_corrs = (dist_corrs .+ transpose(dist_corrs)) / 2

    # 4. Persistent homology (H1)
    d = ripserer(dist_corrs; reps=1, alg=:homology, verbose=false)

    # 5. Build scaffolds
    f_scaffold = zeros(num_regions, num_regions)
    p_scaffold = zeros(num_regions, num_regions)

    for l in 1:length(d[2])
        g = d[2][l]
        pers = persistence(g)
        for rep in representative(g)
            i, j = vertices(rep)
            f_scaffold[i, j] += 1
            f_scaffold[j, i] += 1
            p_scaffold[i, j] += pers
            p_scaffold[j, i] += pers
        end
    end

    # 6. Save
    npzwrite(out_freq, f_scaffold)
    npzwrite(out_pers, p_scaffold)

    ProgressMeter.next!(p)
end

println("Done! Scaffolds saved to: ", output_dir)

## Verify output

In [ ]:
freq_files = readdir(joinpath(output_dir, "freq"))
pers_files = readdir(joinpath(output_dir, "pers"))
println("Freq scaffolds: ", length(freq_files))
println("Pers scaffolds: ", length(pers_files))

# Check a sample
sample = npzread(joinpath(output_dir, "freq", freq_files[1]))
println("Sample shape: ", size(sample))
println("Non-zero entries: ", count(!=(0), sample))
println("Density: ", round(count(!=(0), sample) / length(sample) * 100, digits=2), "%")